# MelodyAI pipeline walkthrough
A synthetic hummed query traced through every stage, using only the public `melodyai` API.

In [ ]:
import numpy as np
from melodyai.demo.generator import generate_catalog
from melodyai.demo.queries import CONDITIONS, excerpt, perturb, synth_query
from melodyai.audio.io import Audio
from melodyai.features.extract import analyze_query, reference_from_notes
from melodyai.indexing.melody_index import MelodyIndex
from melodyai.engine import default_algorithms

songs = generate_catalog(12)
index = MelodyIndex()
for i, m in enumerate(songs):
    index.add(reference_from_notes(f's{i}', m.notes, audio=np.zeros(1)))
print(len(index), 'songs,', len(index.matrix()), 'windows')

## 1. Simulate a hummed query of song 5 (chorus, transposed, slower)

In [ ]:
rng = np.random.default_rng(0)
spec = CONDITIONS['chorus_mid_song']
ex, start = excerpt(songs[5].notes, rng, 6, 9, songs[5].sections, 'chorus')
notes, params = perturb(ex, spec, rng)
y = synth_query(notes, spec, rng)
print(f'excerpt starts at {start:.1f}s', params)

## 2. Pitch tracking and note segmentation

In [ ]:
q = analyze_query(Audio(y, 16000), 'hum', with_chroma=False)
print(q.quality)
print('notes:', np.round(q.notes.pitches, 1))

## 3. Matching with every classical algorithm

In [ ]:
for name, alg in default_algorithms().items():
    if name == 'chroma_dtw':
        continue
    c = alg.search(q, index, 3)
    print(f'{name:12s}', [(x.song_id, round(x.score, 3)) for x in c])